# Selenium 설치

In [5]:
pip install selenium webdriver-manager

Note: you may need to restart the kernel to use updated packages.


# 메인 페이지 내용 크롤링
- 카드 이름

- 카드사

- 캐시백

- 혜택 장소

- 할인

- 해외여부

- 전원실적

## test code

In [8]:
import json
import time
import os
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def clean_filename(filename):
    """파일명 특수문자 제거"""
    return re.sub(r'[\\/*?:"<>|]', "", filename)

def test_crawl_one_basket():
    # 1. 기존 경로 설정 (상위 폴더의 data/cards)
    save_dir = "../data/cards"
    
    # 폴더가 없을 경우 생성하는 안전 장치
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    chrome_options = Options()
    # 테스트이므로 브라우저가 뜨는 것을 확인하기 위해 headless는 끕니다.
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    wait = WebDriverWait(driver, 10)
    
    url = "https://www.card-gorilla.com/card?cate=CHK"
    driver.get(url)

    try:
        # 테스트이므로 '더보기'는 클릭하지 않고 현재 보이는 첫 번째 카드만 잡습니다.
        print("테스트 수집을 시작합니다...")
        time.sleep(3) # 페이지 로딩 대기

        # 2. 카드 리스트 아이템들 가져오기 (이미지의 div.card-container 기준)
        items = driver.find_elements(By.CSS_SELECTOR, "div.card-container")
        
        if not items:
            print("카드를 찾을 수 없습니다. 셀렉터를 확인해주세요.")
            return

        # 3. 첫 번째 카드만 처리 (테스트 로직)
        item = items[0] 
        
        try:
            # --- [ 7가지 항목 한 바구니에 담기 ] ---
            # 이미지 분석을 통한 상대 경로(. 사용) 및 클래스 기반 추출
            
            # 1. 카드 이름 (span.card_name)
            card_name = item.find_element(By.CSS_SELECTOR, "span.card_name").text
            
            # 2. 카드사 (span.card_corp)
            company = item.find_element(By.CSS_SELECTOR, "span.card_corp").text
            
            # 3. 캐시백 (div.sale 내부의 첫 번째 p)
            try:
                cashback = item.find_element(By.CSS_SELECTOR, "div.sale > p:nth-of-type(1)").text
            except:
                cashback = "정보없음"
            
            # 4. 혜택 장소 (div.sale 내부의 두 번째 p)
            try:
                benefit_place = item.find_element(By.CSS_SELECTOR, "div.sale > p:nth-of-type(2)").text
            except:
                benefit_place = "정보없음"
            
            # 5. 할인 (div.sale 내부의 세 번째 p)
            try:
                discount = item.find_element(By.CSS_SELECTOR, "div.sale > p:nth-of-type(3)").text
            except:
                discount = "정보없음"
            
            # 6. 해외여부 (div.ex 내부의 p.in_for)
            try:
                overseas = item.find_element(By.CSS_SELECTOR, "div.ex > p.in_for").text
            except:
                overseas = "정보없음"
            
            # 7. 전월실적 (div.ex 내부의 p.l_mth)
            try:
                performance = item.find_element(By.CSS_SELECTOR, "div.ex > p.l_mth").text
            except:
                performance = "정보없음"

            # 바구니 완성
            card_basket = {
                "card_name": card_name,
                "company": company,
                "cashback": cashback,
                "benefit_place": benefit_place,
                "discount": discount,
                "overseas": overseas,
                "performance": performance
            }
            
            # 4. 파일 저장
            file_name = f"{clean_filename(company)}_{clean_filename(card_name)}.json"
            file_path = os.path.join(save_dir, file_name)

            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(card_basket, f, ensure_ascii=False, indent=4)
            
            print(f"테스트 저장 완료: {file_path}")
            print("데이터 확인:", card_basket)

        except Exception as e:
            print(f"항목 추출 중 오류 발생: {e}")

    finally:
        driver.quit()
        print("테스트 종료.")

if __name__ == "__main__":
    test_crawl_one_basket()

테스트 수집을 시작합니다...
테스트 저장 완료: ../data/cards\케이뱅크_ONE 체크카드.json
데이터 확인: {'card_name': 'ONE 체크카드', 'company': '케이뱅크', 'cashback': '최대\n1.1% 무제한 캐시백', 'benefit_place': '자주 쓰는 곳에서\n5% 캐시백', 'discount': '3번 결제할 때마다\n1,000원 무제한 캐시백', 'overseas': '해외겸용 없음', 'performance': '전월실적 없음'}
테스트 종료.


## 전체 코드 
- 스크롤 후 `카드 더 보기` 클릭 

In [ ]:
import json
import time
import os
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

def clean_filename(filename):
    """파일명 특수문자 제거"""
    return re.sub(r'[\\/*?:"<>|]', "", filename)

def crawl_cards_progressively():
    # 1. 경로 설정
    save_dir = "../data/check_cards"
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    chrome_options = Options()
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    wait = WebDriverWait(driver, 5)
    
    url = "https://www.card-gorilla.com/card?cate=CHK"
    driver.get(url)
    
    scraped_count = 0

    try:
        while True:
            # --- [ 단계 1: 현재 화면에 보이는 카드들 수집 ] ---
            time.sleep(1) # 로딩 대기
            items = driver.find_elements(By.CSS_SELECTOR, "div.card-container")
            
            for item in items:
                try:
                    # 1. 카드 이름 & 2. 카드사 추출
                    card_name = item.find_element(By.CSS_SELECTOR, "span.card_name").text
                    company = item.find_element(By.CSS_SELECTOR, "span.card_corp").text
                    
                    # 파일명 생성 및 중복 확인 (이미 수집한 카드면 건너뜀)
                    file_name = f"{clean_filename(company)}_{clean_filename(card_name)}.json"
                    file_path = os.path.join(save_dir, file_name)
                    
                    if os.path.exists(file_path):
                        continue # 이미 저장된 파일이 있으면 다음 카드로 이동

                    # 3~5. 혜택 정보 (div.sale > p)
                    benefit_elements = item.find_elements(By.CSS_SELECTOR, "div.sale > p")
                    cashback = benefit_elements[0].text if len(benefit_elements) > 0 else "정보없음"
                    benefit_place = benefit_elements[1].text if len(benefit_elements) > 1 else "정보없음"
                    discount = benefit_elements[2].text if len(benefit_elements) > 2 else "정보없음"
                    
                    # 6. 해외여부 & 7. 전월실적
                    try:
                        overseas = item.find_element(By.CSS_SELECTOR, "div.ex > p.in_for").text
                    except: overseas = "정보없음"
                    
                    try:
                        performance = item.find_element(By.CSS_SELECTOR, "div.ex > p.l_mth").text
                    except: performance = "전월실적 없음"

                    # 데이터 저장
                    card_basket = {
                        "card_name": card_name,
                        "company": company,
                        "cashback": cashback,
                        "benefit_place": benefit_place,
                        "discount": discount,
                        "overseas": overseas,
                        "performance": performance
                    }

                    with open(file_path, 'w', encoding='utf-8') as f:
                        json.dump(card_basket, f, ensure_ascii=False, indent=4)
                    
                    scraped_count += 1
                    print(f"[{scraped_count}] 저장 완료: {file_name}")

                except Exception as e:
                    continue # 개별 카드 오류 시 건너뜀

            # --- [ 단계 2: '더 보기' 버튼 클릭 ] ---
            try:
                # 버튼이 화면에 보일 때까지 스크롤 후 클릭
                more_button = driver.find_element(By.CSS_SELECTOR, "a.lst_more")
                driver.execute_script("arguments[0].scrollIntoView();", more_button)
                time.sleep(0.5)
                driver.execute_script("arguments[0].click();", more_button)
                print("\n--- 더 보기 버튼 클릭 (다음 리스트 로딩) ---")
            except NoSuchElementException:
                print("\n모든 카드를 수집했습니다. (더 보기 버튼 없음)")
                break
            except Exception as e:
                print(f"\n더 보기 클릭 중 오류 발생 또는 종료: {e}")
                break

    finally:
        driver.quit()
        print(f"\n총 {scraped_count}개의 새로운 카드 데이터를 수집했습니다.")

if __name__ == "__main__":
    crawl_cards_progressively()


--- 더 보기 버튼 클릭 (다음 리스트 로딩) ---
[1] 저장 완료: KB국민카드_노리2 체크카드(KB Pay).json
[2] 저장 완료: 신한카드_신한카드 SOL트래블 체크.json
[3] 저장 완료: 네이버페이_네이버페이 머니카드.json
[4] 저장 완료: KB국민카드_KB Youth Club 체크카드.json
[5] 저장 완료: KG모빌리언스_모빌리언스카드.json
[6] 저장 완료: 토스뱅크_토스뱅크 체크카드.json
[7] 저장 완료: KB국민카드_트래블러스 체크카드(토심이).json
[8] 저장 완료: 우체국_개이득 체크카드.json
[9] 저장 완료: KB국민카드_KB 틴업 체크카드.json
[10] 저장 완료: MG새마을금고_더나은 체크카드.json
[11] 저장 완료: 하나카드_네이버페이 머니 하나 체크카드.json
[12] 저장 완료: KB국민카드_나라사랑체크카드.json
[13] 저장 완료: KB국민카드_토심이 첵첵 체크카드.json
[14] 저장 완료: 우리카드_카드의정석 오하CHECK.json
[15] 저장 완료: KB국민카드_노리체크카드.json
[16] 저장 완료: 카카오뱅크_카카오뱅크 프렌즈 체크카드.json
[17] 저장 완료: 신한카드_신한카드 Deep Dream 체크.json
[18] 저장 완료: KB국민카드_노리2 체크카드(Global).json
[19] 저장 완료: 신한카드_신한카드 Hey Young 체크.json

--- 더 보기 버튼 클릭 (다음 리스트 로딩) ---
[20] 저장 완료: 하나카드_트래블로그 체크카드.json
[21] 저장 완료: 신한카드_신한카드 On 체크(잔망루피).json
[22] 저장 완료: KB국민카드_직장인보너스체크카드.json
[23] 저장 완료: KB국민카드_트래블러스 체크카드.json
[24] 저장 완료: 하나카드_달달 하나 체크카드.json
[25] 저장 완료: NH농협카드_NH20해봄체크카드.json
[26] 저장 완료: NH농협카드_K-패스카드(체크).json
[27] 저

# 상세 페이지 내용 크롤링
- 주요 혜택과 관련된 데이터 수집

## test code


In [1]:
import json
import time
import os
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def clean_filename(filename):
    """파일명 특수문자 제거"""
    return re.sub(r'[\\/*?:"<>|]', "", filename)

def test_crawl_to_new_folder():
    # 1. 경로 설정
    load_dir = "../data/check_cards"          # 기존 기본 정보가 있는 폴더
    save_dir = "../data/check_cards_benefits" # 새로운 결과물을 저장할 폴더
    
    # 저장 폴더가 없으면 생성
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        print(f"새로운 폴더를 생성했습니다: {save_dir}")
    
    chrome_options = Options()
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    wait = WebDriverWait(driver, 10)
    
    url = "https://www.card-gorilla.com/card?cate=CHK"
    driver.get(url)

    try:
        print("상세 정보 수집 및 새 폴더 저장 테스트를 시작합니다...")
        time.sleep(3)

        # 2. 첫 번째 카드 컨테이너 선택
        item = driver.find_element(By.CSS_SELECTOR, "div.card-container")
        
        # 3. 파일 매칭을 위한 정보 추출
        card_name = item.find_element(By.CSS_SELECTOR, "span.card_name").text
        company = item.find_element(By.CSS_SELECTOR, "span.card_corp").text
        
        file_name = f"{clean_filename(company)}_{clean_filename(card_name)}.json"
        load_path = os.path.join(load_dir, file_name)
        save_path = os.path.join(save_dir, file_name)

        # 4. 기존 폴더에서 기본 데이터 읽기
        if os.path.exists(load_path):
            with open(load_path, 'r', encoding='utf-8') as f:
                card_data = json.load(f)
            print(f"기존 데이터를 불러왔습니다: {load_path}")
        else:
            print(f"경고: {load_dir}에 기존 파일이 없습니다. 기본 정보 없이 진행합니다.")
            card_data = {"card_name": card_name, "company": company}

        # 5. 상세 페이지 이동 및 정보 수집
        detail_url = item.find_element(By.CSS_SELECTOR, "a.b_view").get_attribute("href")
        driver.get(detail_url)
        time.sleep(3)

        benefit_list = []
        try:
            # 주요 혜택 카테고리별 수집
            benefit_items = driver.find_elements(By.CSS_SELECTOR, "div.lst.bene_area dl")
            for dl in benefit_items:
                category = dl.find_element(By.CSS_SELECTOR, "p.txt1").text
                content = dl.find_element(By.TAG_NAME, "i").text
                benefit_list.append({"category": category, "content": content})
        except Exception as e:
            print(f"상세 혜택 수집 중 오류: {e}")

        # 6. 데이터 결합 및 새 폴더에 저장
        card_data["benefit"] = benefit_list
        card_data["source_url"] = detail_url

        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(card_data, f, ensure_ascii=False, indent=4)
        
        print(f"\n[테스트 완료] 새로운 폴더에 저장되었습니다: {save_path}")
        print("수집된 데이터 확인:")
        print(json.dumps(card_data, indent=4, ensure_ascii=False))

    finally:
        driver.quit()

if __name__ == "__main__":
    test_crawl_to_new_folder()

상세 정보 수집 및 새 폴더 저장 테스트를 시작합니다...
기존 데이터를 불러왔습니다: ../data/check_cards\케이뱅크_ONE 체크카드.json

[테스트 완료] 새로운 폴더에 저장되었습니다: ../data/check_cards_benefits\케이뱅크_ONE 체크카드.json
수집된 데이터 확인:
{
    "card_name": "ONE 체크카드",
    "company": "케이뱅크",
    "cashback": "최대\n1.1% 무제한 캐시백",
    "benefit_place": "자주 쓰는 곳에서\n5% 캐시백",
    "discount": "3번 결제할 때마다\n1,000원 무제한 캐시백",
    "overseas": "해외겸용 없음",
    "performance": "전월실적 없음",
    "benefit": [
        {
            "category": "대중교통",
            "content": "[K-패스] 매일 대중교통 타고 교통 요금의 최대 53%까지 돌려받아요"
        },
        {
            "category": "대중교통",
            "content": "대중교통 혜택 추가 3천원 캐시백"
        },
        {
            "category": "캐시백",
            "content": "[모두 다 캐시백] 오프라인 0.6%, 온라인 1.1% 캐시백"
        },
        {
            "category": "캐시백",
            "content": "[여기서 더 캐시백] 자주 쓰는 브랜드에서 최대 5% 캐시백"
        },
        {
            "category": "캐시백",
            "content": "[369 캐시백] 3번 결제할 때마다 1,000원 캐시백"
        },
        {
            "cat

# 신규발급중단 카드 제외 후 재수집

In [9]:
import json
import time
import os
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

def clean_filename(filename):
    """파일명 특수문자 제거"""
    return re.sub(r'[\\/*?:"<>|]', "", filename)

def crawl_active_cards_final():
    # 1. 저장 경로 설정 (정상 카드 전용 폴더)
    save_dir = "../data/check_cards_benefits"
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    chrome_options = Options()
    # chrome_options.add_argument("--headless") # 안정화 후 속도를 위해 주석 해제 가능
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    wait = WebDriverWait(driver, 10)
    
    url = "https://www.card-gorilla.com/card?cate=CHK"
    driver.get(url)

    processed_count = 0 
    saved_count = 0

    try:
        while True:
            time.sleep(2) 
            # 현재 로드된 모든 카드 컨테이너 확보
            items = driver.find_elements(By.CSS_SELECTOR, "div.card-container")
            
            # 새로 추가된 카드들만 슬라이싱하여 처리
            new_items = items[processed_count:]
            if not new_items:
                break
            
            for item in new_items:
                try:
                    # --- [ 필터링: 신규발급중단 체크 ] ---
                    # p.txt_stop 요소가 있으면 수집하지 않고 카운트만 올림
                    stop_els = item.find_elements(By.CSS_SELECTOR, "p.txt_stop")
                    if len(stop_els) > 0:
                        processed_count += 1
                        continue

                    # --- [ 리스트 페이지 정보 추출 ] ---
                    card_name = item.find_element(By.CSS_SELECTOR, "span.card_name").text
                    company = item.find_element(By.CSS_SELECTOR, "span.card_corp").text
                    
                    file_name = f"{clean_filename(company)}_{clean_filename(card_name)}.json"
                    file_path = os.path.join(save_dir, file_name)

                    # 중복 확인: 이미 저장된 파일이 있으면 건너뜀
                    if os.path.exists(file_path):
                        processed_count += 1
                        continue

                    # 혜택 3종 및 조건 2종 추출
                    benefit_els = item.find_elements(By.CSS_SELECTOR, "div.sale > p")
                    cashback = benefit_els[0].text if len(benefit_els) > 0 else "정보없음"
                    benefit_place = benefit_els[1].text if len(benefit_els) > 1 else "정보없음"
                    discount = benefit_els[2].text if len(benefit_els) > 2 else "정보없음"
                    
                    try: overseas = item.find_element(By.CSS_SELECTOR, "div.ex > p.in_for").text
                    except: overseas = "정보없음"
                    
                    try: performance = item.find_element(By.CSS_SELECTOR, "div.ex > p.l_mth").text
                    except: performance = "전월실적 없음"

                    # 상세 페이지 URL (수집 후 JSON에는 담지 않음)
                    detail_url = item.find_element(By.CSS_SELECTOR, "a.b_view").get_attribute("href")

                    # --- [ 상세 페이지 혜택 수집 ] ---
                    driver.execute_script(f"window.open('{detail_url}');")
                    driver.switch_to.window(driver.window_handles[-1])
                    time.sleep(2)

                    detailed_benefits = []
                    try:
                        # 카테고리별 혜택 리스트업
                        benefit_items = driver.find_elements(By.CSS_SELECTOR, "div.lst.bene_area dl")
                        for dl in benefit_items:
                            category = dl.find_element(By.CSS_SELECTOR, "p.txt1").text
                            content = dl.find_element(By.TAG_NAME, "i").text
                            
                            # '유의사항' 카테고리 제외 로직
                            if category != "유의사항":
                                detailed_benefits.append({
                                    "category": category,
                                    "content": content
                                })
                    except: pass

                    # 최종 데이터 구성 (source_url 제외)
                    card_basket = {
                        "card_name": card_name,
                        "company": company,
                        "cashback": cashback,
                        "benefit_place": benefit_place,
                        "discount": discount,
                        "overseas": overseas,
                        "performance": performance,
                        "benefit": detailed_benefits
                    }

                    # 파일 저장
                    with open(file_path, 'w', encoding='utf-8') as f:
                        json.dump(card_basket, f, ensure_ascii=False, indent=4)
                    
                    processed_count += 1
                    saved_count += 1
                    print(f"[{saved_count}] 수집 및 저장 완료: {file_name}")

                    driver.close()
                    driver.switch_to.window(driver.window_handles[0])

                except Exception as e:
                    processed_count += 1
                    if len(driver.window_handles) > 1:
                        driver.close()
                        driver.switch_to.window(driver.window_handles[0])
                    continue

            # --- [ 단계 2: 카드 더 보기 클릭 ] ---
            try:
                more_button = driver.find_element(By.CSS_SELECTOR, "a.lst_more")
                driver.execute_script("arguments[0].scrollIntoView();", more_button)
                time.sleep(1)
                driver.execute_script("arguments[0].click();", more_button)
                print("\n--- 다음 카드 리스트 로드 중 ---")
            except NoSuchElementException:
                print("\n✅ 모든 유효 카드 수집이 완료되었습니다.")
                break

    finally:
        driver.quit()
        print(f"📦 총 저장된 정상 카드 개수: {saved_count}개")

if __name__ == "__main__":
    crawl_active_cards_final()

[1] 수집 및 저장 완료: 케이뱅크_ONE 체크카드.json
[2] 수집 및 저장 완료: KB국민카드_노리2 체크카드(KB Pay).json
[3] 수집 및 저장 완료: 신한카드_신한카드 SOL트래블 체크.json
[4] 수집 및 저장 완료: 네이버페이_네이버페이 머니카드.json
[5] 수집 및 저장 완료: KB국민카드_KB Youth Club 체크카드.json
[6] 수집 및 저장 완료: KG모빌리언스_모빌리언스카드.json
[7] 수집 및 저장 완료: 토스뱅크_토스뱅크 체크카드.json
[8] 수집 및 저장 완료: KB국민카드_트래블러스 체크카드(토심이).json
[9] 수집 및 저장 완료: 우체국_개이득 체크카드.json
[10] 수집 및 저장 완료: KB국민카드_KB 틴업 체크카드.json

--- 다음 카드 리스트 로드 중 ---
[11] 수집 및 저장 완료: 하나카드_네이버페이 머니 하나 체크카드.json
[12] 수집 및 저장 완료: MG새마을금고_더나은 체크카드.json
[13] 수집 및 저장 완료: KB국민카드_토심이 첵첵 체크카드.json
[14] 수집 및 저장 완료: KB국민카드_나라사랑체크카드.json
[15] 수집 및 저장 완료: 우리카드_카드의정석 오하CHECK.json
[16] 수집 및 저장 완료: KB국민카드_노리체크카드.json
[17] 수집 및 저장 완료: 카카오뱅크_카카오뱅크 프렌즈 체크카드.json
[18] 수집 및 저장 완료: 신한카드_신한카드 Deep Dream 체크.json
[19] 수집 및 저장 완료: KB국민카드_노리2 체크카드(Global).json
[20] 수집 및 저장 완료: 신한카드_신한카드 Hey Young 체크.json

--- 다음 카드 리스트 로드 중 ---
[21] 수집 및 저장 완료: KB국민카드_직장인보너스체크카드.json
[22] 수집 및 저장 완료: 하나카드_트래블로그 체크카드.json
[23] 수집 및 저장 완료: KB국민카드_트래블러스 체크카드.json
[24] 수집 및 저장 완료: 신

# 신규발급중단 카드 개수 확인
- 잘 제외했는지 확인하기 위함

In [8]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

def verify_95_suspended_cards():
    chrome_options = Options()
    # 검증을 위해 브라우저가 작동하는 과정을 눈으로 확인하는 것이 좋습니다.
    # chrome_options.add_argument("--headless") 
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    wait = WebDriverWait(driver, 10)
    
    url = "https://www.card-gorilla.com/card?cate=CHK"
    driver.get(url)

    try:
        print("🚀 95개 검증을 위해 모든 카드를 로드합니다. 잠시만 기다려 주세요...")
        
        # 1. 페이지 끝까지 '더 보기' 클릭
        click_count = 0
        while True:
            try:
                # 버튼이 보일 때까지 대기 및 클릭
                more_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.lst_more")))
                driver.execute_script("arguments[0].scrollIntoView();", more_button)
                time.sleep(0.5)
                driver.execute_script("arguments[0].click();", more_button)
                
                click_count += 1
                if click_count % 10 == 0:
                    print(f"ㄴ {click_count}번째 클릭 진행 중...")
            except TimeoutException:
                print("✅ 모든 카드가 로드되었습니다. (더 보기 버튼 없음)")
                break

        # 2. 전수 조사 시작
        items = driver.find_elements(By.CSS_SELECTOR, "div.card-container")
        total_count = len(items)
        
        active_cards = []
        suspended_cards = []

        for item in items:
            card_name = item.find_element(By.CSS_SELECTOR, "span.card_name").text
            company = item.find_element(By.CSS_SELECTOR, "span.card_corp").text
            
            # 신규발급중단 태그 확인
            stop_els = item.find_elements(By.CSS_SELECTOR, "p.txt_stop")
            
            if len(stop_els) > 0:
                suspended_cards.append(f"{company} - {card_name}")
            else:
                active_cards.append(f"{company} - {card_name}")

        # 3. 결과 출력 및 95개 검증
        print("\n" + "="*45)
        print(f"🔍 신규발급중단 카드 검증 리포트")
        print("="*45)
        print(f"❌ 신규 발급 중단 카드 : {len(suspended_cards)}개")
        print(f"✅ 정상 발급 가능 카드 : {len(active_cards)}개")
        print("-" * 45)
        print(f"📊 총 체크카드 합계    : {total_count}개")
        print("="*45)

        # 95개가 맞는지 확인
        if len(suspended_cards) == 95:
            print("✨ 검증 성공: 신규발급중단 카드가 정확히 95개입니다.")
        else:
            print(f"⚠️ 검증 차이 발생: 예상(95개) vs 실제({len(suspended_cards)}개)")
            print("상단 리스트를 확인해 보세요.")

    finally:
        driver.quit()

if __name__ == "__main__":
    verify_95_suspended_cards()

🚀 95개 검증을 위해 모든 카드를 로드합니다. 잠시만 기다려 주세요...
ㄴ 10번째 클릭 진행 중...
ㄴ 20번째 클릭 진행 중...
ㄴ 30번째 클릭 진행 중...
ㄴ 40번째 클릭 진행 중...
✅ 모든 카드가 로드되었습니다. (더 보기 버튼 없음)

🔍 신규발급중단 카드 검증 리포트
❌ 신규 발급 중단 카드 : 95개
✅ 정상 발급 가능 카드 : 372개
---------------------------------------------
📊 총 체크카드 합계    : 467개
✨ 검증 성공: 신규발급중단 카드가 정확히 95개입니다.
